# Sentiment Analysis using PyTorch and NLTK

End-to-end deep learning pipeline for **Mulit-class(3) Sentiment Analysis** (`Negative`, `Neutral`, `Positive`) on Twitter dataset.

### Pipeline Architecture:
1. **Imports & Environment Setup**: PyTorch, NLTK, Pandas, NumPy, and random seeds.
2. **Data Loading & Cleaning**: Reading `train.csv` and `test.csv` from `Sentiment_Dataset.zip` with `latin1` encoding.
3. **Text Preprocessing with NLTK**: Lowercasing, noise/URL removal, NLTK tokenization (`RegexpTokenizer`), stopword filtering, and lemmatization (`WordNetLemmatizer`).
4. **Build Vocabulary**: Custom `Vocabulary` builder class mapping tokens to indices
5. **PyTorch Datasetcustom**: `SentimentDataset` class with `DataLoader` batching.
6. **PyTorch Model Architecture**: Bidirectional LSTM (`SentimentBiLSTM`) neural network.
7. **Training Loop & Validation**: Multi-epoch model training with Cross-Entropy Loss and Adam optimizer.
8. **Evaluation & Metrics**: Performance assessment on test set with Classification Report and Confusion Matrix.
9. **Interactive Inference**: Real-time sentiment prediction function `predict_sentiment()` for custom inputs.

## 1. Imports & Environment Setup

In [1]:
import re
from zipfile import ZipFile
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# random seed
SEED = 632
torch.manual_seed(SEED)
if torch.cuda.is_available:
    torch.cuda.manual_seed_all(SEED)

# device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# filter warnings
import warnings
warnings.filterwarnings(action='ignore', message='FutureWarning')

Using device: cpu


In [2]:
# Download necessary NLTK corpora
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("NLTK resources checked and downloaded successfully!")

NLTK resources checked and downloaded successfully!


## 2. Data Loading & Cleaning

In [3]:
with ZipFile('Sentiment_Dataset.zip', 'r') as z:
    # read train dataset
    with z.open('train.csv', 'r') as f:
        train_df = pd.read_csv(f, encoding='latin1')
    # read test dataset
    with z.open('test.csv', 'r') as f:
        test_df = pd.read_csv(f, encoding='latin1')
        
# getting error due to encoding mismatch so using latin1

In [4]:
# shape of datasets
print(f'Shape of raw train dataset: {train_df.shape}')
print(f'Shape of raw test dataset: {test_df.shape}')

Shape of raw train dataset: (27481, 10)
Shape of raw test dataset: (4815, 9)


In [5]:
train_df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


We are already given refined text as feature *selected_text* but I will create my own refined text

required colums are text and labels

In [6]:
# check for null values and drop them
print(f'Shape of train dataset before droping: {train_df.shape}')
print(f'Shape of test dataset before droping: {test_df.shape}')

train_df = train_df.dropna(axis=0, subset=['text', 'sentiment'])
test_df = test_df.dropna(axis=0, subset=['text', 'sentiment'])

print(f'Shape of train dataset after droping: {train_df.shape}')
print(f'Shape of test dataset after droping: {test_df.shape}')

Shape of train dataset before droping: (27481, 10)
Shape of test dataset before droping: (4815, 9)
Shape of train dataset after droping: (27480, 10)
Shape of test dataset after droping: (3534, 9)


In [11]:
# mapping the sentiments with numerical labels
label2id_mapper = {'negative': 0, 'neutral':1, 'positive':2}
id2label_mapper = {0:'negative', 1:'neutral', 2: 'positive'}

train_df['id'] = train_df['sentiment'].map(label2id_mapper)
test_df['id'] = test_df['sentiment'].map(label2id_mapper)

train_df[['text', 'sentiment', 'id']].head()

,text,sentiment,id
0,"I`d have responded, if I were going",neutral,1
1,Sooo SAD I will miss you here in San Diego!!!,negative,0
2,my boss is bullying me...,negative,0
3,what interview! leave me alone,negative,0
4,"Sons of ****, why couldn`t they put them on t...",negative,0


In [12]:
# distribution of setiments
print('Distribution(count) of sentiments in Train Dataset')
print(train_df['sentiment'].value_counts())

print('\nDistribution(percentage) of sentiments in Train Dataset')
print(f'{(train_df['sentiment'].value_counts()/train_df.shape[0])*100}')

Distribution(count) of sentiments in Train Dataset
sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64

Distribution(percentage) of sentiments in Train Dataset
sentiment
neutral     40.454876
positive    31.229985
negative    28.315138
Name: count, dtype: float64


## 3. Text Preprocessing using NLTK

In [22]:
tokenizer = RegexpTokenizer(r'\w+')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        print('not string type')
        return []

    # lowercase the text
    text = text.lower()
    
    # remove urls and user mentions
    text = re.sub(r'https?://\S+|www.\S+|@\w+', '', text)
    
    # tokenize the text
    tokens = tokenizer.tokenize(text)
    
    # lemmatize and then filter stopwords
    clean_tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words and len(token) > 1 and not token.isdigit()]
    
    return clean_tokens

In [25]:
print(preprocess_text('Hello how are you this is @PrashantKumar check this url https://sjiaojdja. Wanna play a tag game'))

['hello', 'check', 'url', 'wanna', 'play', 'tag', 'game']


## 4. Build Vocabulary